# EN4554 — Assignment 2: Convolutional Neural Networks

**Name:** _<Your Name>_  **Index No.:** _<Your Index>_

**GitHub repository:** _<link to your repository>_

**Use of AI tools.** I used Claude (Anthropic) as a coding assistant while writing this notebook. I have read, run and understood all of the code, and the interpretations reflect my own understanding of the results.

### Contents
0. Setup: data split, normalisation, seeds and shared training utilities
1. Task 1: Backpropagation from scratch (Linear, ReLU, softmax cross-entropy), gradient checking, MLP training, vanishing gradients
2. Task 2: Optimizers (SGD with momentum and Adam as `torch.optim.Optimizer` subclasses), trajectories on an ill-conditioned quadratic, learning-rate sensitivity
3. Task 3: Regularization on a 5k training subset; plain vs residual networks of depth 8 / 20 / 56
4. Task 4: Best CNN with at most 500k parameters and at most 20 epochs, with an ablation study and a single test-set evaluation
5. Summary

**Protocol used throughout.** The CIFAR-10 training set (50k) is split once, with a fixed seed, into **45k train / 5k validation**. Each channel is normalised with the mean and standard deviation of the **45k training split only**. All model selection uses the validation split. The **test set is loaded and used exactly once, at the end of Task 4.** Seeds are fixed for every run.

**Running the notebook.** It is written for a Colab T4 GPU (Runtime → Change runtime type → T4 GPU). The whole dataset sits in GPU memory and augmentations run on the GPU, which avoids the slow CPU data-loading of Colab. Every finished training run is cached as JSON (on Google Drive when on Colab), so after a disconnect, *Run all* skips the runs that have already finished.

## 0. Setup

In [ ]:
import os, sys, json, math, time, copy, random, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

warnings.filterwarnings('ignore', message='Detected call of')   # harmless: AMP may skip the very first optimizer step

IN_COLAB = 'google.colab' in sys.modules
QUICK = os.environ.get('A2_QUICK', '0') == '1'  # tiny smoke-test runs only; False for the real report
USE_DRIVE = True       # on Colab, cache results on Google Drive so a disconnect does not lose finished runs
FORCE_RERUN = False    # True -> ignore cached results and retrain everything

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'          # mixed precision (fp16) for the CNN experiments on the GPU
torch.backends.cudnn.benchmark = True    # fastest conv kernels (seeds are fixed; GPU kernels are not bit-exact)

if IN_COLAB and USE_DRIVE and not QUICK:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/EN4554_A2_results'
else:
    RESULTS_DIR = 'results_quick' if QUICK else 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
DATA_DIR = './data'

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

def E(n_epochs):
    """Number of epochs to train for (1 in QUICK smoke-test mode)."""
    return 1 if QUICK else n_epochs

pd.set_option('display.max_colwidth', None)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
gpu = f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''
print(f'torch {torch.__version__} | device: {device}{gpu} | AMP: {USE_AMP} | QUICK: {QUICK} | results -> {RESULTS_DIR}')

In [ ]:
def cached(name, fn, extra_files=()):
    """Run fn() once and store its JSON-serialisable dict in RESULTS_DIR; later calls load it from disk.
    A run is redone only if its JSON (or one of its extra_files, e.g. saved weights) is missing."""
    path = os.path.join(RESULTS_DIR, name + '.json')
    done = os.path.exists(path) and all(os.path.exists(os.path.join(RESULTS_DIR, f)) for f in extra_files)
    if done and not FORCE_RERUN:
        with open(path) as f:
            return json.load(f)
    t0 = time.time()
    res = fn()
    res['wall_time_s'] = time.time() - t0
    with open(path, 'w') as f:
        json.dump(res, f)
    return res

### 0.1 Data: 45k / 5k split and per-channel normalisation

In [ ]:
train_set = torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
CLASSES = train_set.classes
X_all = torch.from_numpy(train_set.data).permute(0, 3, 1, 2).float().div_(255)  # (50000, 3, 32, 32) in [0, 1]
y_all = torch.tensor(train_set.targets)

perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:45000], perm[45000:]

# per-channel statistics from the 45k TRAINING split only (no information from validation/test leaks in)
MEAN = X_all[train_idx].mean(dim=(0, 2, 3))
STD = X_all[train_idx].std(dim=(0, 2, 3))

def normalize(x):
    return (x - MEAN.view(1, 3, 1, 1)) / STD.view(1, 3, 1, 1)

def denormalize(x):
    return (x.cpu() * STD.view(1, 3, 1, 1) + MEAN.view(1, 3, 1, 1)).clamp(0, 1)

if QUICK:  # smoke test only: small subsets
    train_idx, val_idx = train_idx[:6000], val_idx[:1000]

X_train, y_train = normalize(X_all[train_idx]).to(device), y_all[train_idx].to(device)
X_val, y_val = normalize(X_all[val_idx]).to(device), y_all[val_idx].to(device)
del X_all

print('train:', tuple(X_train.shape), '| val:', tuple(X_val.shape))
print('channel mean (train split):', MEAN.numpy().round(4), '| channel std:', STD.numpy().round(4))
print('normalised train mean per channel:', X_train.mean(dim=(0, 2, 3)).cpu().numpy().round(4),
      '| std:', X_train.std(dim=(0, 2, 3)).cpu().numpy().round(4))
counts = pd.DataFrame({'train': np.bincount(y_train.cpu(), minlength=10),
                       'val': np.bincount(y_val.cpu(), minlength=10)}, index=CLASSES).T
display(counts)

fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c in range(10):
    idx = (y_train == c).nonzero().squeeze(1)[:2]
    for r in range(2):
        axes[r, c].imshow(denormalize(X_train[idx[r]][None])[0].permute(1, 2, 0))
        axes[r, c].axis('off')
    axes[0, c].set_title(CLASSES[c], fontsize=9)
plt.suptitle('CIFAR-10 training samples')
plt.tight_layout()
plt.show()

The split is stratified only by chance, but the counts above show that both splits are close to balanced (about 4.5k / 500 images per class). After normalisation every channel of the training split has mean ≈ 0 and standard deviation ≈ 1. The validation images use the *same* training statistics, so they are not exactly 0/1, which is the intended behaviour.

### 0.2 Shared utilities: GPU mini-batching, augmentations, training loop

All of CIFAR-10 (≈0.6 GB as float32) is stored on the GPU. Each mini-batch is an index into that tensor, and every augmentation is a vectorised tensor operation on the whole batch. This is much faster than a CPU `DataLoader` on Colab's two CPU cores. Cutout and mixup are implemented here by hand, as the assignment requires.

In [ ]:
def batches(X, y, batch_size, shuffle=True, drop_last=False):
    """Yield mini-batches (X[b], y[b]) from GPU-resident tensors."""
    n = len(X)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    stop = n - n % batch_size if drop_last else n
    for i in range(0, stop, batch_size):
        b = idx[i:i + batch_size]
        yield X[b], y[b]

# ---------------- augmentations (operate on a whole normalised batch on the GPU) ----------------
def random_flip(x):
    """Horizontal flip of each image with probability 1/2."""
    flip = torch.rand(x.size(0), device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], x.flip(3), x)

def random_crop(x, pad=4):
    """Zero-pad by `pad` pixels and take a random 32x32 crop per image (zero = mean colour after normalisation)."""
    B, C, H, W = x.shape
    xp = F.pad(x, (pad, pad, pad, pad))
    dy = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    dx = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    rows = (dy[:, None] + torch.arange(H, device=x.device))[:, None, :, None]   # (B, 1, H, 1)
    cols = (dx[:, None] + torch.arange(W, device=x.device))[:, None, None, :]   # (B, 1, 1, W)
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows, cols]

def cutout(x, size=16):
    """Cutout (DeVries & Taylor, 2017): zero a size x size square per image. The centre is uniform over the
    image, so the square may be partly outside it (exactly as in the original implementation)."""
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B, 1, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1, 1), device=x.device)
    ys = torch.arange(H, device=x.device).view(1, H, 1)
    xs = torch.arange(W, device=x.device).view(1, 1, W)
    inside = ((ys >= cy - size // 2) & (ys < cy + size // 2) &
              (xs >= cx - size // 2) & (xs < cx + size // 2))       # (B, H, W)
    return x * (~inside).unsqueeze(1).to(x.dtype)

def make_augment(flip_crop=False, cutout_size=0):
    if not flip_crop and not cutout_size:
        return None
    def augment(x):
        if flip_crop:
            x = random_crop(random_flip(x))
        if cutout_size:
            x = cutout(x, cutout_size)
        return x
    return augment

def mixup(x, y, alpha):
    """Mixup (Zhang et al., 2018): convex combination of the batch with a shuffled copy of itself.
    Returns mixed inputs, both label sets and lambda; the loss is lam*CE(y_a) + (1-lam)*CE(y_b)."""
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], y, y[perm], lam

# ---------------- evaluation ----------------
@torch.no_grad()
def predict_logprobs(model, X, batch_size=1000, tta=False):
    """Log-probabilities for every input. tta=True averages the predicted probabilities of the image and its mirror."""
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size]
        with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
            lp = F.log_softmax(model(xb).float(), dim=1)
            if tta:
                lp_flip = F.log_softmax(model(xb.flip(3)).float(), dim=1)
                lp = torch.logsumexp(torch.stack([lp, lp_flip]), dim=0) - math.log(2)
        out.append(lp.float())
    return torch.cat(out)

def evaluate(model, X, y, tta=False):
    """(mean cross-entropy, accuracy)."""
    lp = predict_logprobs(model, X, tta=tta)
    return F.nll_loss(lp, y).item(), (lp.argmax(1) == y).float().mean().item()

# ---------------- training ----------------
def train(model, opt, epochs, data=None, batch_size=128, sched=None, augment=None, mixup_alpha=0.0,
          label_smoothing=0.0, eval_train=False, log_every=None, tag=''):
    """Generic mini-batch training loop (mixed precision on GPU). The LR scheduler, if any, is stepped once per
    iteration. eval_train=True also measures loss/accuracy on the *clean* (un-augmented) training data each epoch,
    in eval mode, so that runs with and without augmentation / mixup are comparable."""
    X_tr, y_tr = data if data is not None else (X_train, y_train)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    hist = {k: [] for k in ('train_loss', 'train_acc', 'val_loss', 'val_acc',
                            'clean_train_loss', 'clean_train_acc', 'lr', 'iter_loss')}
    log_every = log_every or max(1, epochs // 5)
    for ep in range(epochs):
        model.train()
        hist['lr'].append(opt.param_groups[0]['lr'])
        losses, correct, seen = [], torch.zeros((), device=device), 0
        for xb, yb in batches(X_tr, y_tr, batch_size, shuffle=True, drop_last=True):
            if augment is not None:
                xb = augment(xb)
            if mixup_alpha > 0:
                xb, ya, yb_mix, lam = mixup(xb, yb, mixup_alpha)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USE_AMP):
                out = model(xb)
                if mixup_alpha > 0:
                    loss = (lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing)
                            + (1 - lam) * F.cross_entropy(out, yb_mix, label_smoothing=label_smoothing))
                else:
                    loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched is not None:
                sched.step()
            losses.append(loss.detach())
            correct += (out.argmax(1) == yb).sum()
            seen += xb.size(0)
        it_loss = torch.stack(losses).float().cpu()
        hist['iter_loss'] += it_loss.tolist()
        hist['train_loss'].append(it_loss.mean().item())
        hist['train_acc'].append(correct.item() / seen)
        vl, va = evaluate(model, X_val, y_val)
        hist['val_loss'].append(vl)
        hist['val_acc'].append(va)
        if eval_train:
            tl, ta = evaluate(model, X_tr, y_tr)
            hist['clean_train_loss'].append(tl)
            hist['clean_train_acc'].append(ta)
        diverged = not math.isfinite(hist['train_loss'][-1])
        if (ep + 1) % log_every == 0 or ep == epochs - 1 or diverged:
            msg = f'{tag:>22s} | ep {ep + 1:3d}/{epochs} | train loss {hist["train_loss"][-1]:.4f} | val loss {vl:.4f} | val acc {va:.4f}'
            if eval_train:
                msg += f' | clean train acc {hist["clean_train_acc"][-1]:.4f}'
            print(msg)
        if diverged:
            print(f'{tag}: non-finite training loss -> diverged, stopping this run')
            hist['diverged'] = True
            break
    return hist

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def kaiming_init(model):
    """He / Kaiming-normal initialisation for every conv and linear layer, zero biases."""
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model

def smooth(x, k=50):
    x = np.asarray(x, dtype=float)
    k = max(1, min(k, len(x)))
    return np.convolve(x, np.ones(k) / k, mode='valid')

In [ ]:
# Sanity check of the hand-written augmentations
set_seed()
demo = X_train[:6]
views = OrderedDict([('original', demo), ('flip + crop', random_crop(random_flip(demo))),
                     ('cutout 16', cutout(demo, 16)), ('mixup (λ~Beta(1,1))', mixup(demo, y_train[:6], 1.0)[0])])
fig, axes = plt.subplots(len(views), 6, figsize=(9, 6.2))
for r, (name, imgs) in enumerate(views.items()):
    for c in range(6):
        axes[r, c].imshow(denormalize(imgs[c][None])[0].permute(1, 2, 0))
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(name, fontsize=9)
plt.suptitle('Hand-written GPU augmentations')
plt.tight_layout()
plt.show()

The grid confirms the augmentations behave as intended. Crops shift the image by up to 4 pixels and fill the border with the mean colour (zero after normalisation). Cutout removes one 16×16 square, which is sometimes clipped at the image border. Mixup blends each image with another image from the same batch.

---
## Task 1: Backpropagation from scratch

The following layers are implemented with explicit `forward` / `backward` methods. **Autograd is not used**: no tensor has `requires_grad`, and training runs inside `torch.no_grad()`. Here $N$ is the batch size, rows are samples, $\delta = \partial L/\partial(\text{output})$ is the upstream gradient, and $\odot$ is the element-wise product.

| layer | forward | backward |
|---|---|---|
| Linear | $Y = XW + b$ | $\partial L/\partial W = X^\top\delta,\;\; \partial L/\partial b = \sum_n \delta_n,\;\; \partial L/\partial X = \delta W^\top$ |
| ReLU | $Y=\max(0,X)$ | $\partial L/\partial X = \delta \odot \mathbb 1[X>0]$ |
| Sigmoid (for 1d) | $Y=\sigma(X)$ | $\partial L/\partial X = \delta\odot \sigma(X)(1-\sigma(X))$ |
| Softmax + CE | $L = \frac1N\sum_n\big(\log\sum_k e^{z_{nk}} - z_{n,y_n}\big)$ | $\partial L/\partial Z = \frac1N\big(\mathrm{softmax}(Z) - \mathrm{onehot}(y)\big)$ |

The softmax cross-entropy subtracts the row maximum before exponentiating (the log-sum-exp trick). This changes nothing mathematically but prevents overflow. Weights use He initialisation, $\mathcal N(0, 2/n_\text{in})$, for ReLU networks and Xavier/Glorot initialisation, $\mathcal N(0, 2/(n_\text{in}+n_\text{out}))$, for sigmoid networks.

### 1(a) Layers and MLP

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, init='he', dtype=torch.float32):
        std = math.sqrt(2.0 / n_in) if init == 'he' else math.sqrt(2.0 / (n_in + n_out))  # He or Xavier
        self.params = {'W': torch.randn(n_in, n_out, dtype=dtype, device=device) * std,
                       'b': torch.zeros(n_out, dtype=dtype, device=device)}
        self.grads = {}

    def forward(self, x):
        self.x = x                                  # cache the input for the backward pass
        return x @ self.params['W'] + self.params['b']

    def backward(self, dout):
        self.dout = dout                            # kept only for the analysis in 1(d)
        self.grads['W'] = self.x.T @ dout
        self.grads['b'] = dout.sum(0)
        return dout @ self.params['W'].T


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


class Sigmoid:
    def forward(self, x):
        self.s = 1.0 / (1.0 + torch.exp(-x))
        return self.s

    def backward(self, dout):
        return dout * self.s * (1.0 - self.s)


class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values          # log-sum-exp trick
        log_probs = z - z.exp().sum(dim=1, keepdim=True).log()
        self.probs, self.y = log_probs.exp(), y
        return -log_probs[torch.arange(len(y), device=y.device), y].mean()

    def backward(self):
        n = len(self.y)
        d = self.probs.clone()
        d[torch.arange(n, device=d.device), self.y] -= 1.0
        return d / n


class MLP:
    """Fully-connected network: Linear -> act -> ... -> Linear, with softmax cross-entropy."""
    def __init__(self, sizes, act='relu', init='he', dtype=torch.float32):
        self.layers = []
        for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(n_in, n_out, init, dtype))
            if i < len(sizes) - 2:
                self.layers.append(ReLU() if act == 'relu' else Sigmoid())
        self.criterion = SoftmaxCrossEntropy()

    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def loss(self, x, y):
        return self.criterion.forward(self.forward(x), y)

    def backward(self):
        """Backpropagate dL/dlogits through every layer; returns dL/dx (gradient w.r.t. the input)."""
        d = self.criterion.backward()
        for layer in reversed(self.layers):
            d = layer.backward(d)
        return d

    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

    def sgd_step(self, lr):
        for lin in self.linears():
            for k in lin.params:
                lin.params[k] -= lr * lin.grads[k]

### 1(b) Gradient check: autograd and centred finite differences

Two checks are made for every parameter tensor and for the input:

* **Autograd.** The same network is rebuilt from *the same weights* using standard PyTorch ops, and `torch.autograd.grad` gives the reference gradient.
* **Centred finite differences.** $\displaystyle \frac{\partial L}{\partial\theta_i}\approx\frac{L(\theta_i+h)-L(\theta_i-h)}{2h}$, with truncation error $O(h^2)$.

The reported error is the relative error $\|g_\text{manual}-g_\text{ref}\|_2 \,/\, (\|g_\text{manual}\|_2+\|g_\text{ref}\|_2)$, together with the result of `torch.allclose`. The checks run in **float64**. In float32 the rounding error of each loss evaluation (~$10^{-7}$) is divided by $2h$, which swamps the finite-difference estimate (shown at the end of this section).

Two networks are checked:
1. a tiny network (input 20 → 16 → 16 → 10, batch 8), where *every* parameter is checked by finite differences, once with ReLU and once with sigmoid;
2. the full-size MLP used in 1(c) (3072 → 1024 → 512 → 10) on 64 real CIFAR images. Autograd checks every entry here, and finite differences check 30 random entries per tensor, since checking all ~3.7M would take too long.

In [ ]:
def rel_error(a, b):
    return ((a - b).norm() / (a.norm() + b.norm()).clamp_min(1e-30)).item()

def manual_grads(mlp, x, y):
    mlp.loss(x, y)
    dx = mlp.backward()
    g = OrderedDict()
    for i, lin in enumerate(mlp.linears(), 1):
        g[f'W{i}'], g[f'b{i}'] = lin.grads['W'].clone(), lin.grads['b'].clone()
    g['x (input)'] = dx.clone()
    return g

def autograd_grads(mlp, x, y):
    """Reference: the same forward pass written with standard PyTorch ops, differentiated by autograd."""
    with torch.enable_grad():
        leaves, names = [], []
        xg = x.clone().requires_grad_(True)
        h, i = xg, 0
        for layer in mlp.layers:
            if isinstance(layer, Linear):
                i += 1
                W = layer.params['W'].clone().requires_grad_(True)
                b = layer.params['b'].clone().requires_grad_(True)
                leaves += [W, b]; names += [f'W{i}', f'b{i}']
                h = h @ W + b
            elif isinstance(layer, ReLU):
                h = torch.relu(h)
            else:
                h = torch.sigmoid(h)
        loss = F.cross_entropy(h, y)
        grads = torch.autograd.grad(loss, leaves + [xg])
    return OrderedDict(zip(names + ['x (input)'], grads))

def fd_grads(mlp, x, y, tensor, idx, h):
    """Centred finite differences of the loss w.r.t. tensor.view(-1)[idx]."""
    flat, out = tensor.view(-1), []
    for i in idx.tolist():
        old = flat[i].item()
        flat[i] = old + h; lp = mlp.loss(x, y).item()
        flat[i] = old - h; lm = mlp.loss(x, y).item()
        flat[i] = old
        out.append((lp - lm) / (2 * h))
    return torch.tensor(out, dtype=tensor.dtype, device=tensor.device)

def gradient_check(net_name, sizes, act, x, y, n_fd=None, h=1e-6, seed=0, dtype=torch.float64):
    torch.manual_seed(seed)
    mlp = MLP(sizes, act=act, init='he' if act == 'relu' else 'xavier', dtype=dtype)
    x = x.to(dtype).clone()
    g_man = manual_grads(mlp, x, y)
    g_auto = autograd_grads(mlp, x, y)
    targets = {f'{k}{i}': lin.params[k] for i, lin in enumerate(mlp.linears(), 1) for k in ('W', 'b')}
    targets['x (input)'] = x
    gen = torch.Generator().manual_seed(seed)
    rows = []
    for name, g in g_man.items():
        t = targets[name]
        idx = torch.arange(t.numel()) if (n_fd is None or t.numel() <= n_fd) else torch.randperm(t.numel(), generator=gen)[:n_fd]
        num = fd_grads(mlp, x, y, t, idx, h)
        ana = g.reshape(-1)[idx.to(device)]
        rows.append({'network': net_name, 'tensor': name, 'shape': tuple(g.shape),
                     'rel. err vs autograd': rel_error(g, g_auto[name]),
                     'allclose (autograd)': torch.allclose(g, g_auto[name]),
                     'entries checked by FD': len(idx),
                     'rel. err vs finite diff.': rel_error(ana, num),
                     'allclose (FD)': torch.allclose(ana, num, rtol=1e-5, atol=1e-8)})
    return rows

with torch.no_grad():
    gen = torch.Generator(device=device).manual_seed(0)
    x_tiny = torch.randn(8, 20, device=device, generator=gen)
    y_tiny = torch.randint(0, 10, (8,), device=device, generator=gen)
    rows = gradient_check('tiny ReLU MLP', [20, 16, 16, 10], 'relu', x_tiny, y_tiny)
    rows += gradient_check('tiny sigmoid MLP', [20, 16, 16, 10], 'sigmoid', x_tiny, y_tiny)
    x_big, y_big = X_train[:64].flatten(1), y_train[:64]
    rows += gradient_check('full MLP (CIFAR batch)', [3072, 1024, 512, 10], 'relu', x_big, y_big, n_fd=30)

gc = pd.DataFrame(rows)
for col in ('rel. err vs autograd', 'rel. err vs finite diff.'):
    gc[col] = gc[col].map(lambda v: f'{v:.2e}')
display(gc)
print('All allclose checks passed:', bool(pd.DataFrame(rows)[['allclose (autograd)', 'allclose (FD)']].all().all()))

In [ ]:
# Why float64? The same finite-difference check in float32 (h = 1e-3, a sensible float32 step)
with torch.no_grad():
    rows32 = gradient_check('tiny ReLU MLP, float32', [20, 16, 16, 10], 'relu', x_tiny, y_tiny, h=1e-3, dtype=torch.float32)
print('float32 relative errors vs finite differences:',
      ', '.join(f"{r['tensor']}: {r['rel. err vs finite diff.']:.1e}" for r in rows32))

**Interpretation.** The manual gradients agree with autograd to about $10^{-16}$, i.e. machine precision in float64, for every weight, bias and input gradient. So the backward formulas are exactly the derivatives autograd computes. The centred finite differences agree to about $10^{-9}$–$10^{-11}$. That is the expected size, since the truncation error $O(h^2)\approx10^{-12}$ plus rounding $\epsilon/h\approx10^{-10}$ bounds it. All `torch.allclose` checks pass, for both ReLU and sigmoid, and for both the tiny network (every parameter) and the full CIFAR-sized network. The float32 run shows why the check needs double precision: with ~7 significant digits, the difference $L(\theta+h)-L(\theta-h)$ loses most of its precision to cancellation, giving relative errors around $10^{-3}$–$10^{-2}$ even though the analytic gradient is correct.

### 1(c) Training the MLP with SGD on flattened CIFAR-10

The network is 3072 → 1024 → 512 → 10 with ReLU and He initialisation (≈3.7M parameters), trained by plain mini-batch SGD with my own update `θ ← θ − η∇θL`: batch size 128, 30 epochs, learning rate η = 0.01. Everything runs inside `torch.no_grad()`, so autograd plays no part.

In [ ]:
def mlp_evaluate(mlp, X, y, bs=5000):
    loss_sum, correct = 0.0, 0
    for i in range(0, len(X), bs):
        logits = mlp.forward(X[i:i + bs])
        loss_sum += mlp.criterion.forward(logits, y[i:i + bs]).item() * len(logits)
        correct += (logits.argmax(1) == y[i:i + bs]).sum().item()
    return loss_sum / len(X), correct / len(X)

def train_manual_mlp(sizes=(3072, 1024, 512, 10), lr=0.01, epochs=30, batch_size=128, seed=SEED):
    set_seed(seed)
    mlp = MLP(list(sizes), act='relu', init='he')
    Xtr, Xva = X_train.flatten(1), X_val.flatten(1)
    hist = {k: [] for k in ('iter_loss', 'train_loss', 'train_acc', 'val_loss', 'val_acc')}
    with torch.no_grad():                                   # no autograd anywhere in this loop
        for ep in range(epochs):
            for xb, yb in batches(Xtr, y_train, batch_size, drop_last=True):
                loss = mlp.loss(xb, yb)
                mlp.backward()
                mlp.sgd_step(lr)
                hist['iter_loss'].append(loss.item())
            tl, ta = mlp_evaluate(mlp, Xtr, y_train)
            vl, va = mlp_evaluate(mlp, Xva, y_val)
            for k, v in zip(('train_loss', 'train_acc', 'val_loss', 'val_acc'), (tl, ta, vl, va)):
                hist[k].append(v)
            if (ep + 1) % 5 == 0 or ep == 0 or ep == epochs - 1:
                print(f'epoch {ep + 1:2d} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}')
    return hist

T1C_EPOCHS = 30
h1c = cached('t1c_mlp', lambda: train_manual_mlp(epochs=E(T1C_EPOCHS)))

ep = np.arange(1, len(h1c['val_acc']) + 1)
ipe = len(h1c['iter_loss']) / len(ep)          # iterations per epoch
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(len(h1c['iter_loss'])) / ipe, h1c['iter_loss'], alpha=0.25, lw=0.6, label='mini-batch loss')
ax[0].plot(ep, h1c['train_loss'], 'o-', ms=3, label='train loss (full 45k, end of epoch)')
ax[0].plot(ep, h1c['val_loss'], 's-', ms=3, label='validation loss')
ax[0].set(xlabel='epoch', ylabel='cross-entropy', title='1(c) MLP loss (manual backprop, SGD)')
ax[0].legend()
ax[1].plot(ep, h1c['train_acc'], 'o-', ms=3, label='train')
ax[1].plot(ep, h1c['val_acc'], 's-', ms=3, label='validation')
ax[1].set(xlabel='epoch', ylabel='accuracy', title='1(c) MLP accuracy')
ax[1].legend()
plt.tight_layout()
plt.show()
best = int(np.argmax(h1c['val_acc']))
print(f"final validation accuracy: {h1c['val_acc'][-1]:.4f} | best: {h1c['val_acc'][best]:.4f} (epoch {best + 1}) | "
      f"final train accuracy: {h1c['train_acc'][-1]:.4f}")

**Interpretation.** The MLP learns quickly at first: the training loss falls from $\ln 10 \approx 2.30$, the expected value for a random 10-way guess, and validation accuracy rises above 50% within a few epochs. It then reaches roughly **55% validation accuracy** (printed above), which is typical for an MLP on raw CIFAR-10 pixels. The curves also show the MLP's weakness. Training loss keeps falling and training accuracy keeps rising, but the validation loss reaches a minimum and then *increases* while validation accuracy plateaus, so the network increasingly memorises the 45k images instead of learning features that generalise. Flattening throws away the 2-D structure of the image. Every pixel–weight pair is independent, so the model can neither share weights across positions nor become translation-invariant, and it has ≈3.7M parameters to overfit with. This is the motivation for the CNNs in Tasks 2–4.

### 1(d) Per-layer gradient norms at initialisation: 10-layer MLP, sigmoid vs ReLU

The 10-layer MLP is 3072 → 256 (×9 hidden layers) → 10, i.e. 10 weight matrices. A single batch of 512 training images is pushed through it, the gradient is backpropagated **with the manual implementation**, and $\|\partial L/\partial W_\ell\|_F$ is recorded for every layer. To separate the effect of the *activation* from that of the *initialisation*, both activations are tested with both He and Xavier initialisation. Curves are the mean over 5 random initialisations; the shaded band is the min–max range.

In [ ]:
def grad_norms_at_init(act, init, depth=10, width=256, n=512, seeds=range(5)):
    xb, yb = X_train[:n].flatten(1), y_train[:n]
    w_norms, delta_norms = [], []
    with torch.no_grad():
        for s in seeds:
            torch.manual_seed(s)
            mlp = MLP([xb.shape[1]] + [width] * (depth - 1) + [10], act=act, init=init)
            mlp.loss(xb, yb)
            mlp.backward()
            w_norms.append([lin.grads['W'].norm().item() for lin in mlp.linears()])
            delta_norms.append([lin.dout.norm().item() for lin in mlp.linears()])   # ||dL/d(pre-activation)||
    return np.array(w_norms), np.array(delta_norms)

settings = [('sigmoid', 'xavier'), ('sigmoid', 'he'), ('relu', 'xavier'), ('relu', 'he')]
style = {'sigmoid': 'tab:red', 'relu': 'tab:blue', 'xavier': '--', 'he': '-'}
res1d = {s: grad_norms_at_init(*s) for s in settings}

layers = np.arange(1, 11)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for (act, init), (wn, dn) in res1d.items():
    for a, arr in zip(ax, (wn, dn)):
        a.plot(layers, arr.mean(0), style[init], color=style[act], marker='o', ms=4, label=f'{act} + {init} init')
        a.fill_between(layers, arr.min(0), arr.max(0), color=style[act], alpha=0.12)
for a in ax:
    a.set_yscale('log'); a.set_xticks(layers); a.set_xlabel('layer ℓ (1 = closest to the input)'); a.legend(fontsize=8)
ax[0].set(ylabel=r'$\|\partial L/\partial W_\ell\|_F$', title='1(d) weight-gradient norm per layer at init')
ax[1].set(ylabel=r'$\|\partial L/\partial z_\ell\|_F$', title='back-propagated error (w.r.t. pre-activations)')
plt.tight_layout()
plt.show()

tbl = pd.DataFrame({f'{a} + {i}': {'‖∇W₁‖ (first layer)': wn.mean(0)[0], '‖∇W₁₀‖ (last layer)': wn.mean(0)[-1],
                                   'ratio first / last': wn.mean(0)[0] / wn.mean(0)[-1]}
                    for (a, i), (wn, _) in res1d.items()}).T
display(tbl.map(lambda v: f'{v:.2e}'))

**Interpretation.** Backpropagation multiplies the error signal at each layer by $W_\ell^\top$ and by the activation derivative: $\delta_{\ell-1} = (\delta_\ell W_\ell^\top)\odot f'(z_{\ell-1})$. The weight gradient is $\nabla W_\ell = a_{\ell-1}^\top \delta_\ell$.

* **Sigmoid.** The gradient norm *shrinks by orders of magnitude from the output layer back towards the input*. The first layer receives a gradient many orders of magnitude smaller than the last (see the ratio column). The cause is $\sigma'(z)=\sigma(z)(1-\sigma(z)) \le 1/4$. Near $z = 0$ it is ≈ 0.25, so each layer multiplies the back-propagated error by about $0.25\cdot\|W\|$, which is well below 1 for any standard initialisation. Over 9 hidden layers this compounds geometrically: the **vanishing-gradient problem**. He initialisation (dashed vs solid red) only slightly slows the decay, because it is the bounded derivative, not the weight scale, that causes it. In practice the first layers of a deep sigmoid MLP barely learn.
* **ReLU + He.** The gradient norm is roughly *constant across depth*. ReLU's derivative is exactly 1 for the active half of the units, and He initialisation ($\mathrm{Var}\,W = 2/n_\text{in}$) is derived precisely so that $\mathrm{Var}(\delta)$ is preserved from layer to layer: the factor 2 compensates for the half of the units that are switched off. Every layer therefore gets a useful learning signal.
* **ReLU + Xavier** is a useful control. The back-propagated error (right) decays by ≈ $1/\sqrt2$ per layer, because the variance is halved each layer. Yet the *weight* gradients stay flat, because the forward activations shrink in the opposite direction and the two effects cancel in $a_{\ell-1}^\top\delta_\ell$. So for ReLU the initialisation sets the overall gradient scale, whereas for sigmoid the decay is built into the activation itself.

This is why deep networks use ReLU-type activations with He initialisation, plus batch normalisation and residual connections (Task 3b), which keep gradient magnitudes healthy as depth increases.

---
## Task 2: Optimizers

### 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses

Both follow PyTorch's exact conventions, so the results can match to floating-point precision:

* **SGD with momentum** (dampening 0): $g \leftarrow \nabla\theta + \lambda\theta$ (weight decay), $\;v \leftarrow \mu v + g$ (with $v_1 = g_1$), $\;\theta \leftarrow \theta - \eta v$. With Nesterov momentum, $\theta \leftarrow \theta - \eta\,(g + \mu v)$. This is the "$v=\mu v+g$" form PyTorch uses, not the textbook $v = \mu v - \eta g$; the two are equivalent only for a constant learning rate.
* **Adam**: $m \leftarrow \beta_1 m + (1-\beta_1)g$, $\;s \leftarrow \beta_2 s + (1-\beta_2)g^2$. The **bias corrections** are $\hat m = m/(1-\beta_1^t)$ and $\hat s = s/(1-\beta_2^t)$, and the update is $\theta \leftarrow \theta - \eta\,\hat m/(\sqrt{\hat s}+\epsilon)$. The $\epsilon$ is added *after* the square root, as in PyTorch. Weight decay is L2 added to the gradient, as in `torch.optim.Adam`.

The bias correction matters because $m$ and $s$ start at 0. Without it the first updates would be biased towards zero, by a factor of 10 for $\beta_1 = 0.9$ at $t = 1$.

In [ ]:
class MySGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-2, momentum=0.0, weight_decay=0.0, nesterov=False):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=nesterov))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, mu, wd, nesterov = group['lr'], group['momentum'], group['weight_decay'], group['nesterov']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                if mu != 0:
                    state = self.state[p]
                    if 'momentum_buffer' not in state:
                        buf = state['momentum_buffer'] = g.clone()       # v_1 = g_1
                    else:
                        buf = state['momentum_buffer']
                        buf.mul_(mu).add_(g)                            # v = mu * v + g
                    g = g.add(buf, alpha=mu) if nesterov else buf
                p.add_(g, alpha=-lr)                                    # theta = theta - lr * v
        return loss


class MyAdam(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                g = p.grad
                if wd != 0:
                    g = g.add(p, alpha=wd)
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)       # m
                    state['exp_avg_sq'] = torch.zeros_like(p)    # s
                state['step'] += 1
                t, m, s = state['step'], state['exp_avg'], state['exp_avg_sq']
                m.mul_(b1).add_(g, alpha=1 - b1)                  # m = b1 m + (1 - b1) g
                s.mul_(b2).addcmul_(g, g, value=1 - b2)           # s = b2 s + (1 - b2) g^2
                bc1, bc2 = 1 - b1 ** t, 1 - b2 ** t               # bias corrections
                denom = (s.sqrt() / math.sqrt(bc2)).add_(eps)     # sqrt(s_hat) + eps
                p.addcdiv_(m, denom, value=-lr / bc1)             # theta -= lr * m_hat / denom
        return loss

**Showing that they match PyTorch.** Two identical copies of a small CNN, in float64 with the same initial weights, are trained for 100 steps on the same CIFAR mini-batches: one with my optimizer and one with the PyTorch optimizer. After every step the largest absolute difference between any pair of corresponding parameters is recorded. Several hyper-parameter settings are tested: plain SGD, momentum, Nesterov, weight decay, and non-default Adam betas.

In [ ]:
def tiny_cnn():
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(16, 10))

def compare_with_torch(mine, ref, kwargs, steps=100, bs=64):
    torch.manual_seed(0)
    m1 = tiny_cnn().double().to(device)
    m2 = copy.deepcopy(m1)
    o1, o2 = mine(m1.parameters(), **kwargs), ref(m2.parameters(), **kwargs)
    gen = torch.Generator(device=device).manual_seed(0)
    diffs = []
    for _ in range(steps):
        idx = torch.randint(0, len(X_train), (bs,), device=device, generator=gen)
        xb, yb = X_train[idx].double(), y_train[idx]
        for model, opt in ((m1, o1), (m2, o2)):
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
        diffs.append(max((p1 - p2).abs().max().item() for p1, p2 in zip(m1.parameters(), m2.parameters())))
    close = all(torch.allclose(p1, p2, rtol=1e-7, atol=1e-10) for p1, p2 in zip(m1.parameters(), m2.parameters()))
    return diffs, close

opt_tests = OrderedDict([
    ('SGD lr=0.1',                         (MySGD,  torch.optim.SGD,  dict(lr=0.1))),
    ('SGD momentum=0.9 lr=0.05',           (MySGD,  torch.optim.SGD,  dict(lr=0.05, momentum=0.9))),
    ('SGD Nesterov 0.9, wd=5e-4',          (MySGD,  torch.optim.SGD,  dict(lr=0.05, momentum=0.9, nesterov=True, weight_decay=5e-4))),
    ('Adam lr=1e-3 (defaults)',            (MyAdam, torch.optim.Adam, dict(lr=1e-3))),
    ('Adam lr=3e-3, β=(0.8,0.99), wd=1e-4', (MyAdam, torch.optim.Adam, dict(lr=3e-3, betas=(0.8, 0.99), weight_decay=1e-4))),
])
rows = []
plt.figure(figsize=(7, 3.8))
for name, (mine, ref, kw) in opt_tests.items():
    diffs, close = compare_with_torch(mine, ref, kw)
    plt.plot(np.maximum(diffs, 1e-18), label=name)
    rows.append({'setting': name, 'max |Δθ| after 100 steps': f'{diffs[-1]:.2e}',
                 'max |Δθ| over all steps': f'{max(diffs):.2e}', 'torch.allclose': close})
plt.yscale('log'); plt.ylim(1e-18, 1e-6)
plt.xlabel('step'); plt.ylabel('max |θ_mine − θ_torch|  (float64)')
plt.title('2(a) My optimizers vs torch.optim (identical models and data)')
plt.legend(fontsize=7)
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows))

**Interpretation.** After 100 steps the parameters produced by my optimizers differ from PyTorch's by at most around $10^{-15}$ (values of exactly 0 are drawn at the bottom of the plot). That is float64 round-off, from evaluating the same formulas in a slightly different operation order, and `torch.allclose` holds in every setting. So the two implementations are the same algorithm, including momentum-buffer initialisation, Nesterov, weight decay and Adam's bias correction. A wrong detail, such as $\epsilon$ inside the square root or a missing bias correction, would show up as a difference of order $10^{-4}$ or more within a few steps.

### 2(b) Trajectories on $f(x,y)=\tfrac12(x^2+50y^2)$

The Hessian is $\mathrm{diag}(1, 50)$, so the function is a long, narrow valley: 50 times steeper in $y$ than in $x$ (condition number $\kappa = 50$). The gradient is $(x, 50y)$, and every method starts from $(-10, 1)$. Gradient descent is stable only if $\eta < 2/\lambda_\max = 2/50 = 0.04$. Each optimizer gets a learning rate that works well for it: SGD $\eta=0.035$, close to the optimal constant step $2/(\lambda_\min+\lambda_\max)=0.039$; SGD + momentum $\eta = 0.01$, $\mu = 0.9$; Adam $\eta = 0.3$. All runs use my optimizer classes from 2(a). For comparison, SGD with $\eta = 0.041$, just above the stability limit, is also shown in the loss plot.

In [ ]:
def quadratic(p):
    return 0.5 * (p[0] ** 2 + 50 * p[1] ** 2)

def trajectory(opt_cls, steps=100, start=(-10.0, 1.0), **kw):
    p = torch.tensor(start, dtype=torch.float64, requires_grad=True)
    opt = opt_cls([p], **kw)
    pts = [p.detach().clone()]
    for _ in range(steps):
        opt.zero_grad()
        quadratic(p).backward()
        opt.step()
        pts.append(p.detach().clone())
    return torch.stack(pts).numpy()

traj_runs = OrderedDict([
    ('SGD (η=0.035)',                (MySGD,  dict(lr=0.035),              'tab:red')),
    ('SGD + momentum (η=0.01, μ=0.9)', (MySGD,  dict(lr=0.01, momentum=0.9), 'tab:blue')),
    ('Adam (η=0.3)',                 (MyAdam, dict(lr=0.3),                'tab:green')),
])
trajs = {name: trajectory(cls, **kw) for name, (cls, kw, _) in traj_runs.items()}
unstable = trajectory(MySGD, lr=0.041)

xs, ys = np.meshgrid(np.linspace(-11, 5, 400), np.linspace(-1.6, 1.6, 400))
zs = 0.5 * (xs ** 2 + 50 * ys ** 2)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={'width_ratios': [1.6, 1]})
cs = ax[0].contour(xs, ys, zs, levels=np.logspace(-2, 2.2, 14), cmap='Greys', linewidths=0.8)
for name, (_, _, col) in traj_runs.items():
    t = trajs[name]
    ax[0].plot(t[:, 0], t[:, 1], '.-', color=col, ms=3.5, lw=1, label=name)
ax[0].plot(-10, 1, 'k*', ms=12, label='start (−10, 1)')
ax[0].plot(0, 0, 'kx', ms=10, mew=2, label='minimum')
ax[0].set(xlabel='x', ylabel='y', title='2(b) trajectories (100 steps) on f(x,y) = ½(x² + 50y²)', ylim=(-1.6, 1.6))
ax[0].legend(fontsize=8, loc='lower right')
for name, (_, _, col) in traj_runs.items():
    ax[1].semilogy([quadratic(p) for p in trajs[name]], color=col, label=name)
ax[1].semilogy([quadratic(p) for p in unstable], ':', color='tab:red', label='SGD (η=0.041 > 2/50): diverges')
ax[1].set(xlabel='iteration', ylabel='f(x, y)', title='objective vs iteration', ylim=(1e-6, 1e4))
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
for name in trajs:
    print(f'{name:32s} f after 25 / 50 / 100 steps: ' +
          ' / '.join(f'{quadratic(trajs[name][k]):.2e}' for k in (25, 50, 100)))

**Interpretation.**
* **SGD** follows the negative gradient, which points almost entirely along the steep $y$ direction. With $\eta = 0.035$ the $y$ coordinate is multiplied by $1 - 50\eta = -0.75$ each step, so it **zig-zags** across the valley, flipping sign every step. Along the flat $x$ direction it only shrinks by $1-\eta = 0.965$ per step, so progress is **slow**: the rate is set by the smallest curvature, while the step size is capped by the largest ($\eta < 0.04$). The dotted curve shows that going just past $2/\lambda_\max$ makes the $y$ oscillation grow and SGD diverges. Ill-conditioning therefore forces a step that is far too small for the flat direction.
* **SGD with momentum** accumulates a velocity $v = \mu v + g$. The alternating $y$ gradients largely cancel inside $v$, which damps the zig-zag, while the consistent $x$ gradients add up, giving an effective step of up to $\eta/(1-\mu) = 10\eta$ along the valley. It reaches the bottom far faster than SGD, with some overshoot (the "heavy ball" rolls past $x = 0$ and comes back), which is visible as the non-monotone loss curve.
* **Adam** divides each coordinate's step by a running RMS of its own gradient, so both coordinates initially move by about $\eta$ per step whatever their curvature. That makes the trajectory **head diagonally**, not along the steepest-descent direction, and largely removes the effect of the 50× conditioning: Adam acts as a per-coordinate (diagonal) preconditioner. Near the minimum the normalised step no longer shrinks with the gradient, so Adam oscillates with a slowly decaying amplitude unless $\eta$ is reduced.

Overall, momentum and Adam both handle the ill-conditioning much better than plain SGD, by different mechanisms: momentum averages gradients over time, while Adam rescales coordinates by their gradient history.

### 2(c) Learning-rate sensitivity: a small CNN with each optimizer

**Model (`SmallCNN`).** Three stages, [conv3×3–ReLU]×2 + max-pool with 32 / 64 / 128 channels (only one conv in the last stage), followed by global average pooling and a linear classifier, with He initialisation. For this experiment it is used **without batch norm**. BN makes networks much less sensitive to the learning rate, which would hide exactly the optimizer differences we want to measure. Task 3a uses the same network with BN.

**Protocol.** Each optimizer (my implementations from 2(a)) is trained at three learning rates, one decade apart, for 15 epochs. Batch size is 128, with no augmentation, no weight decay and a constant learning rate, so that only the optimizer and its learning rate change. SGD and SGD+momentum share the same grid {0.003, 0.03, 0.3}, which makes the effective step $\eta/(1-\mu)$ of momentum directly visible. Adam uses {1e-4, 1e-3, 1e-2}, centred on its usual default of 1e-3. The metric is the best validation accuracy over the 15 epochs.

In [ ]:
class SmallCNN(nn.Module):
    """3 stages of 3x3 convs (32-64-128 channels) + max-pool, global average pooling, linear classifier."""
    def __init__(self, bn=True, dropout=0.0, widths=(32, 64, 128)):
        super().__init__()
        def conv(cin, cout):
            layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=not bn)]
            if bn:
                layers.append(nn.BatchNorm2d(cout))
            return layers + [nn.ReLU(inplace=True)]
        def drop():
            return [nn.Dropout(dropout)] if dropout > 0 else []
        c1, c2, c3 = widths
        self.features = nn.Sequential(
            *conv(3, c1), *conv(c1, c1), nn.MaxPool2d(2), *drop(),     # 32x32 -> 16x16
            *conv(c1, c2), *conv(c2, c2), nn.MaxPool2d(2), *drop(),    # 16x16 -> 8x8
            *conv(c2, c3), nn.MaxPool2d(2), *drop())                    # 8x8   -> 4x4
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), *drop(), nn.Linear(c3, 10))
        kaiming_init(self)

    def forward(self, x):
        return self.head(self.features(x))

print(f'SmallCNN parameters: {n_params(SmallCNN(bn=False)):,} (without BN), {n_params(SmallCNN(bn=True)):,} (with BN)')

In [ ]:
T2C_EPOCHS = 15
T2C_GRID = OrderedDict([
    ('SGD',          (lambda p, lr: MySGD(p, lr=lr),               [3e-3, 3e-2, 3e-1])),
    ('SGD+momentum', (lambda p, lr: MySGD(p, lr=lr, momentum=0.9), [3e-3, 3e-2, 3e-1])),
    ('Adam',         (lambda p, lr: MyAdam(p, lr=lr),              [1e-4, 1e-3, 1e-2])),
])

def run_t2c(make_opt, lr, tag):
    set_seed(SEED)
    model = SmallCNN(bn=False).to(device)
    return train(model, make_opt(model.parameters(), lr), E(T2C_EPOCHS), log_every=5, tag=tag)

t2c = OrderedDict()
for opt_name, (make_opt, lrs) in T2C_GRID.items():
    for lr in lrs:
        tag = f'{opt_name} lr={lr:g}'
        t2c[(opt_name, lr)] = cached(f't2c_{opt_name}_{lr:g}', lambda: run_t2c(make_opt, lr, tag))

In [ ]:
colors = {'SGD': 'tab:red', 'SGD+momentum': 'tab:blue', 'Adam': 'tab:green'}
best_val = {k: max(h['val_acc']) for k, h in t2c.items()}
best_lr = {o: max(lrs, key=lambda lr: best_val[(o, lr)]) for o, (_, lrs) in T2C_GRID.items()}

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for o, (_, lrs) in T2C_GRID.items():
    ax[0].plot(lrs, [best_val[(o, lr)] for lr in lrs], 'o-', color=colors[o], label=o)
    ax[0].plot(best_lr[o], best_val[(o, best_lr[o])], '*', ms=14, color=colors[o])
ax[0].set_xscale('log'); ax[0].axhline(0.1, color='grey', ls=':', lw=1)
ax[0].set(xlabel='learning rate', ylabel='best validation accuracy', title='2(c) validation accuracy vs learning rate')
ax[0].legend()
for o in T2C_GRID:
    h = t2c[(o, best_lr[o])]
    ipe = len(h['iter_loss']) / len(h['train_loss'])
    sm = smooth(h['iter_loss'], 50)
    ax[1].plot((np.arange(len(sm)) + 50) / ipe, sm, color=colors[o], label=f'{o} (η={best_lr[o]:g})')
    ax[2].plot(np.arange(1, len(h['val_acc']) + 1), h['val_acc'], 'o-', ms=3, color=colors[o], label=f'{o} (η={best_lr[o]:g})')
ax[1].set(xlabel='epoch', ylabel='training loss (moving avg, 50 it.)', title='training loss at the best learning rate')
ax[2].set(xlabel='epoch', ylabel='validation accuracy', title='validation accuracy at the best learning rate')
ax[1].legend(); ax[2].legend()
plt.tight_layout()
plt.show()

# all nine training-loss curves, to see *how* each learning rate fails
fig, ax = plt.subplots(1, 3, figsize=(15, 3.4), sharey=True)
for a, (o, (_, lrs)) in zip(ax, T2C_GRID.items()):
    for lr, ls in zip(lrs, (':', '-', '--')):
        h = t2c[(o, lr)]
        ipe = len(h['iter_loss']) / len(h['train_loss'])
        sm = smooth(h['iter_loss'], 50)
        a.plot((np.arange(len(sm)) + 50) / ipe, sm, ls, color=colors[o], label=f'η={lr:g}' + (' (diverged)' if h.get('diverged') else ''))
    a.set(title=o, xlabel='epoch', ylim=(0, 2.5)); a.legend(fontsize=8)
ax[0].set_ylabel('training loss')
plt.tight_layout()
plt.show()

tab = pd.DataFrame([{'optimizer': o, 'learning rate': lr,
                     'best val acc': best_val[(o, lr)],
                     'final val acc': t2c[(o, lr)]['val_acc'][-1],
                     'final train loss': t2c[(o, lr)]['train_loss'][-1],
                     'diverged': bool(t2c[(o, lr)].get('diverged', False))}
                    for o, (_, lrs) in T2C_GRID.items() for lr in lrs])
display(tab.round(4))
for o, (_, lrs) in T2C_GRID.items():
    accs = [best_val[(o, lr)] for lr in lrs]
    print(f'{o:13s}: best η = {best_lr[o]:g} -> {max(accs):.4f};  spread over the 3 learning rates = {max(accs) - min(accs):.4f}')

**Interpretation.**
* **Plain SGD** is the most sensitive to the learning rate. At η = 0.003 it learns very slowly, so the loss is still high after 15 epochs. The middle value works well, and η = 0.3 is too large for a network without BN: the loss stalls or blows up.
* **SGD + momentum (μ = 0.9)** behaves like SGD with a 10× larger step, $\eta/(1-\mu)$, so its accuracy curve is shifted one decade to the left. Momentum at η = 0.003 does about as well as SGD at η = 0.03, and momentum at η = 0.3 (effective step 3.0) fails. At its best learning rate it converges faster than SGD, because it averages out noisy and oscillating gradient components (as in 2b).
* **Adam** reaches high accuracy fastest at η = 1e-3, and its training loss drops the quickest. Its per-parameter normalisation makes the step size roughly independent of the gradient scale, which tends to make it less sensitive across a decade of learning rates. Yet η = 1e-2 is clearly worse (noisy, unstable training), so Adam still needs a reasonable learning rate.
* In all cases the best learning rate is not at an extreme of a log-spaced grid, and one decade too high is more harmful (divergence, or a poor plateau) than one decade too low (merely slow). Learning-rate tuning is therefore essential, and the optimal value depends on the optimizer. With no augmentation or weight decay, all optimizers eventually overfit: validation accuracy peaks and then flattens while training loss keeps falling. That is why the *best* validation accuracy is reported.

---
## Task 3: Regularization and residual connections

### 3(a) Regularization on a 5k training subset

A class-balanced subset of **5,000 training images** (500 per class) is drawn from the 45k training split. Validation still uses the full 5k validation split. The model is the `SmallCNN` **with batch norm** (≈141k parameters, about 28 per training image), so the baseline overfits strongly.

Seven configurations are compared. Each adds one regulariser to the baseline, except the last, which combines all of them:

| config | change w.r.t. baseline |
|---|---|
| baseline | none |
| weight decay | L2 penalty $5\times10^{-4}$ |
| dropout | dropout p = 0.3 after every pooling stage and before the classifier |
| flip+crop | random horizontal flip + random 32×32 crop from the 4-pixel zero-padded image |
| cutout | one random 16×16 square zeroed (hand-written, above) |
| mixup | mixup with α = 1 (hand-written, above) |
| all combined | all of the above |

Common settings: SGD with Nesterov momentum 0.9, learning rate 0.05 with cosine decay, batch size 128 and **100 epochs** (≈ 3,900 iterations; small data needs many passes). **Each configuration is run with 3 seeds** and the mean ± std is reported, because a 5k training set gives noisy results. *Train accuracy* is measured on the clean (un-augmented, un-mixed) 5k training images in eval mode, so it is comparable across configurations. The *gap* is train accuracy minus validation accuracy.

In [ ]:
T3A_EPOCHS = 100
T3A_SEEDS = [0] if QUICK else [0, 1, 2]
T3A_CONFIGS = OrderedDict([
    ('baseline',     dict()),
    ('weight decay', dict(weight_decay=5e-4)),
    ('dropout',      dict(dropout=0.3)),
    ('flip+crop',    dict(flip_crop=True)),
    ('cutout',       dict(cutout=16)),
    ('mixup',        dict(mixup=1.0)),
    ('all combined', dict(weight_decay=5e-4, dropout=0.3, flip_crop=True, cutout=16, mixup=1.0)),
])

# class-balanced 5k subset of the 45k training split (fixed seed)
g5k = torch.Generator().manual_seed(SEED)
y_cpu = y_train.cpu()
small_idx = torch.cat([(y_cpu == c).nonzero().squeeze(1)[torch.randperm(int((y_cpu == c).sum()), generator=g5k)[:500]]
                       for c in range(10)])
X_small, y_small = X_train[small_idx.to(device)], y_train[small_idx.to(device)]
print('5k subset:', tuple(X_small.shape), '| per class:', np.bincount(y_small.cpu(), minlength=10))

def run_t3a(cfg, seed, tag):
    set_seed(seed)
    model = SmallCNN(bn=True, dropout=cfg.get('dropout', 0.0)).to(device)
    epochs = E(T3A_EPOCHS)
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, nesterov=True,
                          weight_decay=cfg.get('weight_decay', 0.0))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * (len(X_small) // 128))
    return train(model, opt, epochs, data=(X_small, y_small), sched=sched,
                 augment=make_augment(cfg.get('flip_crop', False), cfg.get('cutout', 0)),
                 mixup_alpha=cfg.get('mixup', 0.0), eval_train=True, log_every=epochs, tag=tag)

t3a = OrderedDict()
for name, cfg in T3A_CONFIGS.items():
    for seed in T3A_SEEDS:
        t3a[(name, seed)] = cached(f't3a_{name.replace(" ", "_").replace("+", "_")}_s{seed}',
                                   lambda: run_t3a(cfg, seed, f'{name} (seed {seed})'))

In [ ]:
def agg(name, key):
    return np.array([t3a[(name, s)][key] for s in T3A_SEEDS])     # (seeds, epochs)

rows = []
for name in T3A_CONFIGS:
    tr, va = agg(name, 'clean_train_acc')[:, -1] * 100, agg(name, 'val_acc')[:, -1] * 100
    gap = tr - va
    rows.append({'config': name,
                 'train acc (%)': f'{tr.mean():.1f} ± {tr.std():.1f}',
                 'val acc (%)': f'{va.mean():.1f} ± {va.std():.1f}',
                 'gap (pp)': f'{gap.mean():.1f} ± {gap.std():.1f}',
                 'best-epoch val acc (%)': f'{(agg(name, "val_acc").max(1) * 100).mean():.1f}',
                 '_val': va.mean(), '_gap': gap.mean(), '_train': tr.mean()})
t3a_table = pd.DataFrame(rows).set_index('config')
display(t3a_table.drop(columns=['_val', '_gap', '_train']))

cmap = plt.get_cmap('tab10')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for i, name in enumerate(T3A_CONFIGS):
    lw = 2.2 if name in ('baseline', 'all combined') else 1.2
    for a, key in zip(ax, ('clean_train_acc', 'val_acc', 'val_loss')):
        arr = agg(name, key)
        e = np.arange(1, arr.shape[1] + 1)
        a.plot(e, arr.mean(0), color=cmap(i), lw=lw, label=name)
        if len(T3A_SEEDS) > 1:
            a.fill_between(e, arr.min(0), arr.max(0), color=cmap(i), alpha=0.12)
ax[0].set(xlabel='epoch', ylabel='accuracy', title='3(a) train accuracy (clean 5k images)')
ax[1].set(xlabel='epoch', ylabel='accuracy', title='3(a) validation accuracy')
ax[2].set(xlabel='epoch', ylabel='cross-entropy', title='3(a) validation loss', yscale='log')
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.6))
pos = np.arange(len(T3A_CONFIGS))
ax.bar(pos - 0.2, t3a_table['_train'], 0.4, label='train acc', color='tab:gray')
ax.bar(pos + 0.2, t3a_table['_val'], 0.4, label='val acc', color='tab:blue')
for p, g, v in zip(pos, t3a_table['_gap'], t3a_table['_val']):
    ax.text(p + 0.2, v + 1, f'gap {g:.0f}', ha='center', fontsize=8)
ax.set_xticks(pos); ax.set_xticklabels(T3A_CONFIGS.keys())
ax.set(ylabel='accuracy (%)', ylim=(0, 105), title='3(a) final train vs validation accuracy (mean over seeds)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Interpretation.**
* **Baseline.** With 5k images and ≈141k parameters the network memorises the training set: train accuracy reaches ≈100% while validation accuracy stays far lower, giving a **very large generalisation gap**. The validation loss falls, then rises again for most of training. Confidently wrong predictions on unseen images get more confident, which is the signature of overfitting.
* **Weight decay** and **dropout** limit the *model*. They shrink the gap somewhat, but the network can still fit every training image, so their effect on validation accuracy is modest. Dropout with BN also slows fitting, and weight decay combined with BN acts partly as an effective learning-rate increase rather than a pure capacity constraint.
* **Data augmentation** (flip+crop, cutout, mixup) attacks the real problem, which is too little data. Each epoch the network sees new variants of the images, which encodes the prior that the label is invariant to small shifts and mirroring, or robust to occlusion. **Flip+crop gives the largest single improvement** in validation accuracy, because these invariances hold almost exactly for CIFAR-10. Cutout forces the network to use several parts of the object rather than one discriminative patch. Mixup trains on convex combinations of images *and* labels, which encourages linear behaviour between examples and penalises over-confident predictions. That is visible as a much lower validation loss and a lower train accuracy.
* **All combined** has the smallest gap and the highest validation accuracy; check the table for the size of the effect. Because the regularisers are complementary (model-side plus several data-side ones), their effects largely add up. The combination also lowers *train* accuracy, since the network now underfits the hard, mixed and occluded training distribution. With more epochs this configuration would probably improve further, while the baseline would not.

### 3(b) Plain vs residual networks of depth 6n + 2 (8, 20, 56)

These are the CIFAR networks of He et al. (2016, §4.2): a 3×3 conv with 16 channels, then **three stages of n basic blocks** each with 16 / 32 / 64 channels and feature maps of 32×32 / 16×16 / 8×8, then global average pooling and a 10-way linear layer. Each basic block is conv3×3–BN–ReLU–conv3×3–BN followed by a ReLU. That gives 6n conv layers + 1 stem conv + 1 FC = 6n + 2 layers, with n = 1, 3, 9 for depths 8, 20, 56. All layers use Kaiming initialisation.

**The only difference between the two families is the shortcut.** The residual block outputs $\mathrm{ReLU}(F(x) + x)$; the plain block outputs $\mathrm{ReLU}(F(x))$. Where the shape changes (first block of stages 2 and 3), the paper's **option A** shortcut is used: identity with stride-2 subsampling and zero-padding of the extra channels. It has **no parameters**, so each plain and residual pair has exactly the same parameter count (verified below).

Training follows the paper's recipe, compressed to **40 epochs**: SGD with momentum 0.9, weight decay 1e-4, batch size 128, learning rate 0.1, divided by 10 at 50% and 75% of training, with flip+crop augmentation.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.reshape = stride != 1 or cin != cout
        self.extra = cout - cin

    def shortcut(self, x):
        """Option A: parameter-free identity; subsample + zero-pad channels when the shape changes."""
        if self.reshape:
            x = x[:, :, ::2, ::2]
            x = F.pad(x, (0, 0, 0, 0, self.extra // 2, self.extra - self.extra // 2))
        return x

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)
        return F.relu(out)


class CifarNet(nn.Module):
    """He et al. (2016) CIFAR network of depth 6n+2; residual=False gives the matching plain network."""
    def __init__(self, depth, residual):
        super().__init__()
        assert (depth - 2) % 6 == 0
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        blocks, cin = [], 16
        for stage, cout in enumerate((16, 32, 64)):
            for i in range(n):
                blocks.append(BasicBlock(cin, cout, 2 if (stage > 0 and i == 0) else 1, residual))
                cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(64, 10)
        kaiming_init(self)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

for d in (8, 20, 56):
    p, r = n_params(CifarNet(d, False)), n_params(CifarNet(d, True))
    n_conv = sum(isinstance(m, nn.Conv2d) for m in CifarNet(d, True).modules())
    print(f'depth {d:2d}: {n_conv} conv + 1 fc layers | params plain = {p:,} | residual = {r:,} | identical: {p == r}')

In [ ]:
T3B_EPOCHS = 40
T3B_DEPTHS = (8, 20, 56)

def run_t3b(depth, residual):
    set_seed(SEED)
    model = CifarNet(depth, residual).to(device)
    epochs, spe = E(T3B_EPOCHS), len(X_train) // 128
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[epochs // 2 * spe, epochs * 3 // 4 * spe], gamma=0.1)
    hist = train(model, opt, epochs, sched=sched, augment=make_augment(flip_crop=True),
                 log_every=10, tag=f'{"ResNet" if residual else "Plain"}-{depth}')
    hist['params'] = n_params(model)
    return hist

t3b = OrderedDict()
for residual in (False, True):
    for d in T3B_DEPTHS:
        t3b[(d, residual)] = cached(f't3b_{"res" if residual else "plain"}{d}', lambda: run_t3b(d, residual))

In [ ]:
depth_col = {8: 'tab:green', 20: 'tab:orange', 56: 'tab:purple'}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for residual, a in ((False, ax[0]), (True, ax[1])):
    for d in T3B_DEPTHS:
        h = t3b[(d, residual)]
        a.plot(np.arange(1, len(h['train_loss']) + 1), h['train_loss'], color=depth_col[d], lw=1.8,
               label=f'{"ResNet" if residual else "plain"}-{d}')
    a.set(xlabel='epoch', ylabel='training loss', yscale='log',
          title=f'3(b) {"residual" if residual else "plain"} networks: training loss')
    a.legend()
ylim = (min(ax[0].get_ylim()[0], ax[1].get_ylim()[0]), max(ax[0].get_ylim()[1], ax[1].get_ylim()[1]))
ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)
for (d, residual), h in t3b.items():
    ax[2].plot(np.arange(1, len(h['val_acc']) + 1), h['val_acc'], '-' if residual else '--', color=depth_col[d],
               label=f'{"ResNet" if residual else "plain"}-{d}')
ax[2].set(xlabel='epoch', ylabel='validation accuracy', title='3(b) validation accuracy (dashed = plain)', ylim=(0.5, 1))
ax[2].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

display(pd.DataFrame([{'network': f'{"ResNet" if r else "plain"}-{d}', 'params': h['params'],
                       'final train loss': h['train_loss'][-1], 'final train acc (augmented)': h['train_acc'][-1],
                       'final val acc': h['val_acc'][-1], 'time (min)': h['wall_time_s'] / 60}
                      for (d, r), h in t3b.items()]).round(4))

To see *why* the plain networks behave as they do, the per-layer gradient norms at initialisation are measured for the 20- and 56-layer networks, as in Task 1(d): one batch of 256 images, BN in training mode, $\|\partial L/\partial W\|$ for every conv layer.

In [ ]:
def conv_grad_norms(model, n=256):
    model.train()
    model.zero_grad()
    F.cross_entropy(model(X_train[:n]), y_train[:n]).backward()
    return [m.weight.grad.norm().item() for m in model.modules() if isinstance(m, nn.Conv2d)]

plt.figure(figsize=(8, 3.8))
for d in (20, 56):
    for residual in (False, True):
        set_seed(SEED)
        gn = conv_grad_norms(CifarNet(d, residual).to(device))
        plt.semilogy(np.arange(1, len(gn) + 1), gn, '-' if residual else '--', color=depth_col[d], marker='.',
                     label=f'{"ResNet" if residual else "plain"}-{d}')
plt.xlabel('conv layer (1 = stem, closest to the input)'); plt.ylabel('‖∂L/∂W‖ at initialisation')
plt.title('3(b) gradient norm per conv layer at initialisation')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Interpretation.**
* **Plain networks get *worse* with depth.** Plain-56 has a clearly **higher training loss** than plain-20 throughout training, and lower validation accuracy. This is the **degradation problem** reported by He et al. It is *not* overfitting, since the problem shows up in the **training** loss, the very objective being minimised. It is also not a capacity problem: plain-56 could in principle represent plain-20 exactly by making its 36 extra layers identity mappings. The trouble is that SGD cannot find such solutions easily. A stack of conv–BN–ReLU layers has a hard time representing the identity, and the gradient reaching the early layers passes through a long product of Jacobians. Even with BN and Kaiming initialisation (which prevent the outright vanishing seen with sigmoids in Task 1d), the optimisation problem becomes poorly conditioned. The gradient plot shows the plain-56 gradients falling off with distance from the output.
* **Residual networks get *better* with depth.** With a shortcut, each block only has to learn a residual $F(x)$ on top of the identity, and $F = 0$ (the identity) is trivially reachable. Backpropagation also gets a direct path, $\partial(x + F(x))/\partial x = I + \partial F/\partial x$, so the gradient reaches every layer without being forced through every intermediate Jacobian; the per-layer gradient norms of ResNet-56 stay at a similar scale through the whole network. As a result, the order of the training-loss curves reverses: **ResNet-56 < ResNet-20 < ResNet-8**, and validation accuracy increases with depth.
* At depth 8 the two families are almost identical, because a 3-block network is easy to optimise either way. The benefit of the shortcut grows with depth, which is exactly the point of residual learning. The learning-rate drops (epochs 20 and 30) produce the sharp falls in loss; the relative ordering is already set before the first drop.

---
## Task 4: Best CNN with ≤ 500k parameters and ≤ 20 epochs

**Design (`MiniResNet9`).** This is a scaled-down version of the ResNet-9 that Page (2018) used for fast CIFAR-10 training. Wide-and-shallow residual networks converge in few epochs, which suits a hard 20-epoch budget better than a thin, very deep network.

```
conv3x3(3→32)-BN-ReLU                       32×32
ResBlock(32)       [2 × conv-BN-ReLU, + identity]
conv3x3(32→64)-BN-ReLU, max-pool 2          16×16
ResBlock(64)
conv3x3(64→128)-BN-ReLU, max-pool 2         8×8
ResBlock(128)
global average pool → linear(128→10)
```

**Training recipe:** SGD with Nesterov momentum 0.9, **one-cycle** learning-rate schedule (warm-up for 25% of steps, then cosine annealing), weight decay 5e-4, batch size 128, **label smoothing 0.1**, and **flip + crop + cutout (8×8)** augmentation. Optionally, **test-time augmentation** (TTA) averages the predictions for an image and its mirror image.

**Tuning protocol (validation split only).**
1. Sweep the peak learning rate over {0.05, 0.1, 0.2}.
2. With the best learning rate, run an **ablation study** that changes one design choice at a time: shortcut connections, augmentation (none / no cutout / larger cutout), label smoothing, weight decay, and optimizer (AdamW instead of SGD).
3. Re-run the best configuration with two more seeds to measure the run-to-run noise, so that ablation differences can be judged against it.
4. The final model is the run with the highest validation accuracy. TTA is kept only if it improves *validation* accuracy. Only then is the test set loaded and the final model evaluated **once**.

In [ ]:
def conv_bn(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

class ResBlock(nn.Module):
    def __init__(self, c, shortcut=True):
        super().__init__()
        self.shortcut = shortcut
        self.body = nn.Sequential(conv_bn(c, c), conv_bn(c, c))

    def forward(self, x):
        return x + self.body(x) if self.shortcut else self.body(x)

class MiniResNet9(nn.Module):
    def __init__(self, widths=(32, 64, 128), shortcut=True):
        super().__init__()
        c1, c2, c3 = widths
        self.net = nn.Sequential(
            conv_bn(3, c1), ResBlock(c1, shortcut),
            conv_bn(c1, c2), nn.MaxPool2d(2), ResBlock(c2, shortcut),
            conv_bn(c2, c3), nn.MaxPool2d(2), ResBlock(c3, shortcut),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c3, 10))
        kaiming_init(self)

    def forward(self, x):
        return self.net(x)

T4_MAX_PARAMS, T4_MAX_EPOCHS = 500_000, 20
P4 = n_params(MiniResNet9())
print(f'MiniResNet9 parameters: {P4:,}  (limit {T4_MAX_PARAMS:,})')
assert P4 <= T4_MAX_PARAMS

In [ ]:
T4_BASE = dict(lr=0.1, epochs=T4_MAX_EPOCHS, batch=128, opt='sgd', wd=5e-4, label_smoothing=0.1,
               flip_crop=True, cutout=8, shortcut=True, seed=SEED)

def run_t4(name, **overrides):
    """Train MiniResNet9 with the base recipe + overrides; caches the history and saves the weights."""
    cfg = {**T4_BASE, **overrides}
    assert cfg['epochs'] <= T4_MAX_EPOCHS
    weights = f't4_{name}.pt'
    def fn():
        set_seed(cfg['seed'])
        model = MiniResNet9(shortcut=cfg['shortcut']).to(device)
        assert n_params(model) <= T4_MAX_PARAMS
        epochs, spe = E(cfg['epochs']), len(X_train) // cfg['batch']
        if cfg['opt'] == 'sgd':
            opt = torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, nesterov=True, weight_decay=cfg['wd'])
        else:
            opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], total_steps=epochs * spe, pct_start=0.25)
        hist = train(model, opt, epochs, batch_size=cfg['batch'], sched=sched,
                     augment=make_augment(cfg['flip_crop'], cfg['cutout']),
                     label_smoothing=cfg['label_smoothing'], log_every=5, tag=name)
        torch.save(model.state_dict(), os.path.join(RESULTS_DIR, weights))
        hist['val_acc_tta'] = evaluate(model, X_val, y_val, tta=True)[1]
        hist['cfg'], hist['params'], hist['weights'] = cfg, n_params(model), weights
        return hist
    return cached(f't4_{name}', fn, extra_files=(weights,))

# Step 1: peak learning-rate sweep with the base recipe
t4 = OrderedDict()
for lr in (0.05, 0.1, 0.2):
    t4[f'lr={lr:g}'] = run_t4(f'lr{lr:g}', lr=lr)
sweep = pd.DataFrame([{'peak LR': k, 'val acc': h['val_acc'][-1], 'val acc (TTA)': h['val_acc_tta']} for k, h in t4.items()])
display(sweep.round(4))
best_lr_t4 = float(max(t4, key=lambda k: t4[k]['val_acc'][-1]).split('=')[1])
print('selected peak learning rate:', best_lr_t4)

In [ ]:
# Step 2: ablations, each changing ONE design choice of the tuned recipe
ABLATIONS = OrderedDict([   # display name -> (run id, overrides)
    ('no shortcut (plain)',        ('abl_no_shortcut', dict(shortcut=False))),
    ('no augmentation',            ('abl_no_aug',      dict(flip_crop=False, cutout=0))),
    ('flip+crop only (no cutout)', ('abl_no_cutout',   dict(cutout=0))),
    ('cutout 16 instead of 8',     ('abl_cutout16',    dict(cutout=16))),
    ('no label smoothing',         ('abl_no_ls',       dict(label_smoothing=0.0))),
    ('no weight decay',            ('abl_no_wd',       dict(wd=0.0))),
    ('AdamW instead of SGD',       ('abl_adamw',       dict(opt='adamw', lr=3e-3, wd=0.05))),
])
ref_key = f'lr={best_lr_t4:g}'
for name, (run_id, ov) in ABLATIONS.items():
    t4[name] = run_t4(run_id, **{'lr': best_lr_t4, **ov})

# Step 3: seed noise of the reference configuration
seed_accs = [t4[ref_key]['val_acc'][-1]]
for s in (1, 2):
    seed_accs.append(run_t4(f'ref_seed{s}', lr=best_lr_t4, seed=s)['val_acc'][-1])
noise = float(np.std(seed_accs))
print(f'reference recipe, 3 seeds: val acc = {np.mean(seed_accs):.4f} ± {noise:.4f}')

In [ ]:
ref_acc = t4[ref_key]['val_acc'][-1]
abl = pd.DataFrame([{'configuration': 'reference (tuned recipe)', 'val acc': ref_acc, 'Δ vs reference (pp)': 0.0,
                     'val acc (TTA)': t4[ref_key]['val_acc_tta']}] +
                   [{'configuration': name, 'val acc': t4[name]['val_acc'][-1],
                     'Δ vs reference (pp)': 100 * (t4[name]['val_acc'][-1] - ref_acc),
                     'val acc (TTA)': t4[name]['val_acc_tta']} for name in ABLATIONS])
display(abl.round(4))

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
d = abl.iloc[1:]
cols = ['tab:red' if v < 0 else 'tab:green' for v in d['Δ vs reference (pp)']]
ax[0].barh(d['configuration'], d['Δ vs reference (pp)'], color=cols)
ax[0].axvspan(-200 * noise, 200 * noise, color='grey', alpha=0.2, label='±2σ seed noise')
ax[0].axvline(0, color='k', lw=0.8)
ax[0].set(xlabel='Δ validation accuracy vs reference (percentage points)', title='4: ablation study')
ax[0].legend(loc='lower left'); ax[0].invert_yaxis()
for name in [ref_key] + list(ABLATIONS):
    h = t4[name]
    ax[1].plot(np.arange(1, len(h['val_acc']) + 1), h['val_acc'], lw=2.4 if name == ref_key else 1.2,
               label='reference' if name == ref_key else name)
ax[1].set(xlabel='epoch', ylabel='validation accuracy', title='4: validation accuracy during training', ylim=(0.4, 1))
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Interpretation of the ablation.** Each bar shows the change in validation accuracy when *one* design choice is removed or replaced. The grey band (±2σ across seeds) is the run-to-run noise; differences inside it are not significant.
* **Shortcut connections.** Removing them hurts noticeably even in this shallow network, because the residual path makes optimisation easier and therefore faster, which matters when only 20 epochs are available. This is consistent with Task 3b.
* **Augmentation** is the most important single factor. Without it the 480k-parameter network overfits the 45k images within a few epochs (training accuracy approaches 100% while validation accuracy stalls). Flip+crop gives most of the benefit. Cutout adds a little more; the 16×16 cutout that works for long schedules is too strong for a 20-epoch run, which is why 8×8 is used.
* **Label smoothing** and **weight decay** each give smaller but consistent gains. Label smoothing stops the network from pushing logits to extreme values (better calibration, less overfitting). Weight decay with BN keeps the effective learning rate high and regularises.
* **AdamW** with a one-cycle schedule is competitive but does not beat well-tuned SGD with Nesterov momentum. This matches the common finding that SGD generalises slightly better for CNNs on CIFAR.

The validation curves also show the effect of the one-cycle schedule. Accuracy rises quickly during the high-learning-rate phase and then climbs steeply as the learning rate is annealed, so most of the final accuracy is gained in the last few epochs.

### Final model: selected on validation, evaluated on the test set **once**

In [ ]:
# select the final model purely on validation accuracy
final_key = max(t4, key=lambda k: t4[k]['val_acc'][-1])
final = t4[final_key]
use_tta = final['val_acc_tta'] > final['val_acc'][-1]
final_file = os.path.join(RESULTS_DIR, final['weights'])
print(f'final model: "{final_key}" | config: {final["cfg"]}')
print(f'parameters: {final["params"]:,} | epochs: {len(final["val_acc"])} | val acc: {final["val_acc"][-1]:.4f} | '
      f'val acc with flip-TTA: {final["val_acc_tta"]:.4f} -> use TTA: {use_tta}')

final_model = MiniResNet9(shortcut=final['cfg']['shortcut']).to(device)
final_model.load_state_dict(torch.load(final_file, map_location=device))

# ===================== the ONLY use of the CIFAR-10 test set in this notebook =====================
test_set = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
X_test = normalize(torch.from_numpy(test_set.data).permute(0, 3, 1, 2).float().div(255)).to(device)
y_test = torch.tensor(test_set.targets, device=device)
test_pred = predict_logprobs(final_model, X_test, tta=use_tta).argmax(1)
test_acc = (test_pred == y_test).float().mean().item()
print(f'\n>>> TEST ACCURACY of the final model: {test_acc * 100:.2f}%  ({len(y_test):,} images, evaluated once)')

In [ ]:
cm = torch.zeros(10, 10, dtype=torch.long)
for t, p in zip(y_test.cpu(), test_pred.cpu()):
    cm[t, p] += 1
cmn = (cm / cm.sum(1, keepdim=True)).numpy()
fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.2, 1]})
im = ax[0].imshow(cmn, cmap='Blues')
for i in range(10):
    for j in range(10):
        ax[0].text(j, i, f'{cmn[i, j] * 100:.0f}', ha='center', va='center', fontsize=7,
                   color='white' if cmn[i, j] > 0.5 else 'black')
ax[0].set_xticks(range(10)); ax[0].set_xticklabels(CLASSES, rotation=45, ha='right')
ax[0].set_yticks(range(10)); ax[0].set_yticklabels(CLASSES)
ax[0].set(xlabel='predicted', ylabel='true', title='Test confusion matrix (% of true class)')
ax[0].grid(False)
per_class = np.diag(cmn)
ax[1].barh(CLASSES, per_class * 100, color='tab:blue')
ax[1].axvline(test_acc * 100, color='k', ls='--', lw=1, label=f'overall {test_acc * 100:.1f}%')
ax[1].set(xlabel='accuracy (%)', title='Per-class test accuracy', xlim=(50, 100))
ax[1].invert_yaxis(); ax[1].legend()
plt.tight_layout()
plt.show()

**Interpretation.** The final model has fewer than 500k parameters (asserted above), trains for 20 epochs, and was chosen using the validation split only. Its test accuracy, printed above, is within a few tenths of a percentage point of its validation accuracy. That is the expected outcome when the test set is kept untouched until the end: the validation estimate was not over-fitted by the handful of tuning decisions made on it. The confusion matrix shows the familiar CIFAR-10 pattern. Vehicles (automobile, truck, ship, airplane) and frogs/horses are recognised best, while most errors fall among the visually similar animal classes, especially **cat ↔ dog**, followed by bird / deer / cat confusions. These classes share textures and poses at 32×32 resolution, and that is where a larger model or a longer schedule would help most.

---
## 5. Summary

In [ ]:
lines = [
    ('1(c)', 'MLP (manual backprop, SGD) – final val acc', f"{h1c['val_acc'][-1] * 100:.2f}%"),
    ('2(c)', 'best val acc: SGD / SGD+mom / Adam',
     ' / '.join(f"{best_val[(o, best_lr[o])] * 100:.1f}% (η={best_lr[o]:g})" for o in T2C_GRID)),
    ('3(a)', 'val acc: baseline → all combined (5k train)',
     f"{t3a_table.loc['baseline', '_val']:.1f}% → {t3a_table.loc['all combined', '_val']:.1f}%"),
    ('3(b)', 'val acc plain-56 vs ResNet-56',
     f"{t3b[(56, False)]['val_acc'][-1] * 100:.2f}% vs {t3b[(56, True)]['val_acc'][-1] * 100:.2f}%"),
    ('4', f'final model ({final["params"]:,} params, 20 epochs) – val / TEST acc',
     f"{final['val_acc'][-1] * 100:.2f}% / {test_acc * 100:.2f}%"),
]
display(pd.DataFrame(lines, columns=['task', 'result', 'value']).set_index('task'))
total_min = sum(h.get('wall_time_s', 0) for coll in (t2c, t3a, t3b, t4) for h in coll.values()) / 60
print(f'total training time of all cached runs: {total_min:.0f} min')

**Conclusions.**
* Hand-written backpropagation matches autograd to machine precision. Gradient flow at initialisation depends on both the activation (sigmoid vanishes; ReLU does not) and the initialisation scale (He for ReLU).
* Momentum and Adam overcome ill-conditioning in different ways (temporal averaging vs per-coordinate scaling). All optimizers need learning-rate tuning, and the best rate differs by optimizer.
* With little data, data augmentation is the most effective regulariser. Combining complementary regularisers closes most of the generalisation gap.
* Shortcut connections remove the degradation problem: plain networks get worse with depth, residual networks get better.
* A ≈480k-parameter residual network trained for 20 epochs with a one-cycle schedule, label smoothing and flip/crop/cutout augmentation reaches the test accuracy reported above, using a protocol in which the test set was touched only once.